### Николай Жмуров, М25-555

# Поиск общих ключевых точек на нескольких изображениях

**Домашнее задание.** Дано несколько изображений организмов одного типа.
Нужно найти ключевые точки, соответствующие одним и тем же локальным особенностям
объекта на всех изображениях, и визуализировать их с одинаковыми номерами.



####Загрузка изображений из Google Drive

Скачиваем папку с изображениями по публичной ссылке с помощью библиотеки `gdown`.

In [ ]:
# Импорт необходимых библиотек
# Системные
import os               # работа с файловой системой (создание папок, переменные окружения)
import time             # замер времени выполнения детекторов и матчеров
import warnings         # управление предупреждениями
warnings.filterwarnings("ignore")  # отключаем шумные warnings (например, от torch.hub)

# Загрузка данных
import gdown            # скачивание папок/файлов с Google Drive

# Работа с путями
from pathlib import Path

# Компьютерное зрение и математика
import cv2              # OpenCV: детекторы (AKAZE, SIFT), матчеры, RANSAC, гомография
import numpy as np      # массивы, линейная алгебра, работа с координатами

# Визуализация
import matplotlib.pyplot as plt  # вывод изображений и графиков

# Таблицы
import pandas as pd     # сводные таблицы по парам/точкам

# Глубокое обучение (для XFeat, SuperPoint, LightGlue)
import torch            # PyTorch: тензоры, нейросетевые модели
import torch.nn as nn   # базовые модули (для возможной тонкой настройки)

# Утилиты
from collections import defaultdict  # словари со значением по умолчанию (для union-find и группировок)

# --- Проверка окружения ---
print(f"OpenCV:      {cv2.__version__}")
print(f"NumPy:       {np.__version__}")
print(f"Pandas:      {pd.__version__}")
print(f"PyTorch:     {torch.__version__}")
print(f"CUDA:        {'доступна' if torch.cuda.is_available() else 'недоступна'}")
print(f"Устройство:  {'cuda' if torch.cuda.is_available() else 'cpu'}")




In [ ]:
# ID папки Google Drive
FOLDER_ID = "1SkIjE_XUW9CAFDfQutDJrzoZlPh1JMfm"
DATA_DIR = "images"

os.makedirs(DATA_DIR, exist_ok=True)

# Скачиваем всю папку
gdown.download_folder(
    id=FOLDER_ID,
    output=DATA_DIR,
    quiet=False,
    use_cookies=False
)

# gdown сохраняет без расширения; добавляем .jpg
for p in Path(DATA_DIR).iterdir():
    if p.is_file() and p.suffix == "":
        new_p = p.with_suffix(".jpg")
        p.rename(new_p)

# Список скачанных изображений
exts = (".jpg",)
paths = sorted(
    p for p in Path(DATA_DIR).rglob("*")
    if p.suffix.lower() in exts
)

print(f"\nСкачано изображений: {len(paths)}\n")
for p in paths:
    print(" ", p)

In [ ]:
# Просмотр загруженных изображений
n = len(paths)
cols = min(n, 3)
rows = (n + cols - 1) // cols

plt.figure(figsize=(3 * cols, 2 * rows))
for i, p in enumerate(paths, 1):
    img = cv2.imread(str(p))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    plt.subplot(rows, cols, i)
    plt.imshow(img)
    plt.title(p.name, fontsize=9)
    plt.axis("off")
plt.tight_layout()
plt.show()

## Обзор методов поиска ключевых точек

В работе изначально предлагаются четыре классических метода детекции и описания локальных особенностей, доступных в OpenCV: **SIFT**, **ORB**, **AKAZE** и **BRISK**. Все они решают одну задачу — найти на изображении устойчивые точки и построить их числовые дескрипторы, но делают это принципиально разными способами.

| Метод | Тип дескриптора | Инвариантность | Особенности |
|-------|-----------------|----------------|-------------|
| **SIFT** | Вещественный (128 чисел) | Масштаб, поворот, частично освещение | Классика, высокая точность, но медленный и чувствителен к размытию и низкому контрасту |
| **ORB** | Бинарный (256 бит) | Масштаб, поворот | Очень быстрый, но на слабоконтрастных и малогекстурных изображениях находит мало точек |
| **AKAZE** | Бинарный (MLDB) | Масштаб, поворот, частично аффинные искажения | Нелинейное масштабное пространство, устойчив к размытию и изменению контраста |
| **BRISK** | Бинарный | Масштаб, поворот | Быстрый, но чувствителен к шуму и слабому контрасту |



## Описание задачи и особенности набора изображений

Дано 6 изображений одного организма — рачка *Bosmina longirostris* — полученных под микроскопом. Набор обладает рядом особенностей, которые напрямую влияют на выбор метода поиска ключевых точек:

1. **Разный масштаб и разрешение.** Изображения отличаются более чем в 4 раза по площади:
   - 2592×1944, 3005×2513, 1608×1944, 1959×1760, 2170×2094, 778×632 px.
   Такое расхождение требует от детектора устойчивости к изменению масштаба.

2. **Разная цветовая модель.** Часть изображений получена в **градациях серого** (чёрно-белые снимки микроскопа), часть — **цветные** (вероятно, с другой камеры или в другом режиме съёмки). Это означает, что цветовые каналы не несут стабильной информации, и все методы должны работать с интенсивностью (grayscale), а не с цветом.

3. **Разная экспозиция и контраст.** Часть снимков пересвечена, часть — недодержана. Хитиновые покровы, щетинки и антеннулы имеют низкий градиент яркости относительно фона.

4. **Зеркальность.** Часть изображений снята с противоположной стороны объекта (аналогично фотографии человека слева и справа). Это значит, что одни и те же биологические структуры на таких снимках зеркально отражены.

5. **Слабая текстура.** Поверхность рачка не имеет богатой текстуры, как, например, фасады зданий. Детектор должен находить устойчивые точки на ограниченном числе контрастных структур: сложный глаз, основание антеннул, границы створок раковины, крупные щетинки.



## Сравнение методов применительно к задаче

| Критерий задачи | SIFT | ORB | AKAZE | BRISK |
|-----------------|------|-----|-------|-------|
| Слабоконтрастные структуры | Пропускает из-за `contrastThreshold` | Находит мало точек | **Находит благодаря низкому порогу и нелинейной диффузии** | Чувствителен к шуму |
| Разный масштаб (до 4×) | Устойчив | Устойчив | **Устойчив (нелинейное масштабное пространство)** | Устойчив |
| Разная экспозиция | Частично устойчив | Слабо устойчив | **Устойчив (диффузия подавляет шум, сохраняет границы)** | Слабо устойчив |
| Цветные + ЧБ изображения | Работает с grayscale | Работает с grayscale | **Работает с grayscale** | Работает с grayscale |
| Слабая текстура | Мало точек | Мало точек | **Больше точек за счёт MLDB-дескриптора** | Мало точек |
| Скорость сопоставления | Медленно (float, 128D) | Быстро (binary) | **Быстро (binary MLDB)** | Быстро (binary) |
| Зеркальность | Не инвариантен | Не инвариантен | **Не инвариантен** | Не инвариантен |

### Ключевые наблюдения

- **SIFT** даёт точные дескрипторы, но на слабоконтрастных микроскопических снимках теряет значительную часть точек, а сопоставление вещественных дескрипторов высокого разрешения занимает секунды.
- **ORB** и **BRISK** быстрые, но на малогекстурных изображениях с низким контрастом находят слишком мало точек для устойчивого сопоставления.
- **AKAZE** сочетает преимущества: нелинейное масштабное пространство (устойчивость к масштабу и размытию), нелинейная диффузия (сохранение границ при подавлении шума) и бинарный MLDB-дескриптор (быстрое сопоставление). С пониженным порогом `threshold=0.001` он находит точки даже на слабоконтрастных участках.
- **Общее ограничение всех четырёх методов** — отсутствие инвариантности к зеркальному отражению. Это учитывается отдельно на этапе сопоставления (стратегия flip-matching), а не на уровне выбора детектора.



## Вывод

Для данного набора изображений микроскопии *Bosmina longirostris* теоритически **оптимальным методом является AKAZE**. Он обеспечивает наилучший баланс между количеством находимых точек на слабоконтрастных структурах, устойчивостью к разнице масштаба и экспозиции, а также скоростью сопоставления бинарных дескрипторов. Ограничение по зеркальности решается на следующем этапе — при сопоставлении дескрипторов.

## Поиск ключевых точек: AKAZE

У нас есть шесть изображений одного организма *Bosmina longirostris*. По совокупности критериев (слабоконтрастные структуры, разный масштаб, разная экспозиция, слабая текстура, смешанные цветные и ЧБ снимки) в качестве основного метода выбран **AKAZE**.

Сначала запускаем его «как есть» — с параметрами, адаптированными под микроскопию, но без какой-либо предобработки изображений. Это даст нам **базовую точку отсчёта**: мы увидим, сколько точек метод находит на каждом изображении и как распределены эти точки по площади кадра.

Для этого:

1. Создаём детектор AKAZE с пониженным порогом (`threshold=0.001`) и нелинейной диффузией — эти параметры критичны для слабоконтрастных микроскопических изображений.
2. Прогоняем его по всем шести изображениям через `run_method`.
3. Замеряем время работы на каждом изображении.
4. Визуализируем найденные ключевые точки через `show_keypoints`.
5. Сохраняем результат в общий словарь `results` под именем `AKAZE_base`.

На этом шаге мы **не боремся с зеркальностью, не выравниваем масштаб и не трогаем контраст** — цель в том, чтобы зафиксировать исходное поведение метода и понять, где именно он проседает. Это станет основой для последующих шагов: предобработки, настройки параметров и учёта зеркальности.

## Детекция AKAZE

Используем очень низкий порог `threshold=0.001`, чтобы получить
максимум точек на каждом изображении. Это даст сопоставлению больше
материала. Никакой предобработки и RANSAC — чистая проверка метода.

Параметры:
- `threshold=0.001` — низкий порог, много точек;
- `nOctaves=4` — стандартное число октав;
- `nOctaveLayers=4` — стандартное число слоёв;
- `diffusivity=KAZE_DIFF_PM_G2` — сохранение границ;
- `descriptor_type=MLDB` — стандартный бинарный дескриптор.

In [ ]:
# Загрузка изображений в словарь images
images = {str(p): cv2.imread(str(p)) for p in paths}
print(f"Загружено изображений: {len(images)}")
for k, v in images.items():
    print(" ", Path(k).name, v.shape)

In [ ]:
# Функции для детекции и визуализации
def run_method(detector, images, method_name="Method"):
    """Прогоняет детектор по всем изображениям."""
    results = {}
    for path, img in images.items():
        start = time.time()
        kp, des = detector.detectAndCompute(img, None)
        elapsed = time.time() - start
        results[path] = (kp, des, elapsed)
        print(f"  {method_name} | {Path(path).name}: найдено {len(kp)} точек, время {elapsed:.3f} с")
    return results


def show_keypoints(image, keypoints, title="Keypoints", max_kp=None):
    """Отображает ключевые точки случайными цветами."""
    img_draw = image.copy()
    if max_kp:
        keypoints = keypoints[:max_kp]
    rng = np.random.default_rng(42)
    colors = rng.integers(50, 255, size=(len(keypoints), 3)).tolist()
    for kp, color in zip(keypoints, colors):
        x, y = int(kp.pt[0]), int(kp.pt[1])
        cv2.circle(img_draw, (x, y), 4, color, -1)
    plt.figure(figsize=(10, 8))
    plt.imshow(cv2.cvtColor(img_draw, cv2.COLOR_BGR2RGB))
    plt.title(f"{title} (показано {len(keypoints)} точек)")
    plt.axis("off")
    plt.show()

In [ ]:
# Создание детектора AKAZE
akaze_base = cv2.AKAZE_create(
    threshold=0.001,
    nOctaves=4,
    nOctaveLayers=4,
    diffusivity=cv2.KAZE_DIFF_PM_G2,
    descriptor_type=cv2.AKAZE_DESCRIPTOR_MLDB
)

print("AKAZE с параметрами по умолчанию (threshold=0.001)")
results_akaze_base = run_method(akaze_base, images, "AKAZE_base")

# Визуализация точек
for path, (kp, des, _) in results_akaze_base.items():
    show_keypoints(images[path], kp, title=f"AKAZE_base: {Path(path).name}")

Вывод по базовому запуску AKAZE

### Итоговая таблица

| Изображение | Точек | Время, с | Оценка |
|---|---|---|---|
| 1 | 10 086 | 0.65 | много, но приемлемо |
| 2 | 30 060 | 1.13 | избыток, много шума |
| 3 | 28 792 | 0.56 | избыток |
| 4 | 23 545 | 0.59 | избыток |
| 5 | 13 998 | 0.65 | много |
| 6 | **20** | 0.04 | очень мало |

### Результаты

**1. Огромный разброс по числу точек — от 20 до 30 060, то есть в 1500 раз.** Это главная проблема базового запуска. С таким дисбалансом сопоставление не будет успешным: изображение 6 не даст достаточного числа кандидатов для матчинга, а изображения 2–4, наоборот, вызовут десятки тысяч шумовых совпадений.

**2. Порог `threshold=0.001` слишком низкий.** При таком пороге AKAZE цепляется за всё подряд: шум сенсора, артефакты JPEG, пылинки на препарате, микроскопические градиенты освещённости. Большинство этих точек нестабильны — на другом снимке того же рачка их там уже не будет.

**3. Изображение 6 — особая проблема.** Оно самое маленькое (778×632, в 4–5 раз меньше остальных по площади). AKAZE строит нелинейное масштабное пространство через октавы: при малом разрешении верхние октавы почти пустые, а на нижней остаётся мало контрастных структур. Плюс, вероятно, снимок недодержан или пересвечен — низкий контраст хитиновых покровов не даёт детектору зацепиться.


Базовый запуск **не годится для сопоставления** в текущем виде. Прежде чем переходить к матчингу и поиску общих точек, нужно привести изображения к сопоставимому виду и ограничить число слабых точек:

1. **Предобработка:** перевод в grayscale, выравнивание локального контраста через CLAHE, приведение всех изображений к общему разрешению (1200 px по большей стороне), повышение локальной резкости (unsharp mask) — для компенсации размытия после апскейла маленького изображения.
2. **Учёт зеркальности:** построение двух версий каждого изображения — оригинала и горизонтально отражённой — для последующего flip-matching.
3. **Ограничение числа точек:** `max_points=5000` в `cv2.AKAZE_create` выравнивает изображения между собой и отсекает слабые, шумовые точки.
4. **Стратегия матчинга:** cross Check-матчинг с порогом по расстоянию Хэмминга, мягкий порог RANSAC (15 px), выбор модели (гомография или фундаментальная матрица) в зависимости от ракурса пары.

После этих шагов детектор может дать сбалансированный набор точек, пригодный для устойчивого сопоставления.


## Для справки:

Виды матчинга дескрипторов

---
Brute-Force (BFMatcher). Для каждой точки первого изображения перебираются все точки второго и ищется ближайшая по расстоянию. Для бинарных дескрипторов (AKAZE, ORB, BRISK) расстояние — Хэмминга (число различающихся бит), для вещественных (SIFT) — евклидово (NORM_L2). Просто, надёжно, но O(N×M) — при 5000×5000 это 25 млн сравнений, работает за секунды.

kNN-матчинг (k=2). Для каждой точки ищутся два ближайших соседа на втором изображении. Если лучший сосед существенно ближе второго — соответствие надёжное. Если расстояние почти одинаковое — точка «не уверена», и пара отбрасывается. Это основа ratio test (Lowe's ratio): d1 < 0.75 × d2.

CrossCheck. Вместо k=2 делается симметричная проверка: точка A→B считается соответствием только если B — ближайший сосед для A, и A — ближайший сосед для B. Даёт компактный и чистый набор пар без порогов и коэффициентов.

Ratio test vs CrossCheck. Ratio строже: отбрасывает всё, что вызывает сомнения. CrossCheck мягче: оставляет пары, где обе стороны «уверены друг в друге». На слабоконтрастных снимках, где дескрипторы похожи, ratio режет слишком много — CrossCheck сохраняет больше правильных кандидатов.

Порог по абсолютному расстоянию. После CrossCheck пары сортируются по расстоянию, и отсекаются все с расстоянием больше max_dist (например, 100 из 256 бит). Это отсекает случайные соответствия с большим расхождением дескрипторов.

---

## RANSAC
RANSAC (RANdom SAmple Consensus) — итеративный алгоритм для поиска модели, согласованной с большинством данных, при наличии большого числа выбросов.

Как работает:

Случайно выбирается минимальный набор точек, необходимый для построения модели (4 точки — для гомографии, 8 — для фундаментальной матрицы).

По этому набору строится модель.

Все остальные точки проверяются: если они согласуются с моделью (ошибка < порога ransacReprojThreshold), они считаются inliers.

Шаги повторяются N раз (обычно 1000–10000 итераций).

Побеждает модель с наибольшим числом inliers.

Что даёт:

Устойчивость к выбросам: даже если 90% совпадений — мусор, RANSAC найдёт модель по оставшимся 10%.

Финальный набор inliers — те точки, которые согласуются с геометрией. Именно они идут дальше.

Параметры:

ransacReprojThreshold — максимальная ошибка, при которой точка считается inlier. Для гомографии 5 px жёстко, 15–20 px мягко. Мягкий порог нужен, когда изображения сильно различаются по масштабу и ракурсу.

maxIters — число итераций. Больше итераций → выше шанс найти правильную модель.

confidence — целевая вероятность, что найдена правильная модель (обычно 0.995–0.999).

Минимальное решение. RANSAC при N точек может выдать модель, даже если все совпадения случайны. Для гомографии достаточно 4 точки, для F-матрицы — 8. Поэтому 4–6 inliers у AKAZE — это артефакт, а не результат: RANSAC просто нашёл 4 точки, случайно согласующиеся с гомографией. Настоящее сопоставление даёт десятки-сотни inliers.

---
## Гомография vs фундаментальная матрица
Гомография (H, 3×3). Описывает проективное преобразование между двумя плоскостями. Работает, когда объект плоский, или когда камера вращается вокруг оптического центра без смещения, или когда сцена снята с одного ракурса.

8 степеней свободы (одна нормировка).

Минимальный набор для RANSAC: 4 точки.

Даёт сильные ограничения: прямые остаются прямыми, параллельные — параллельными.

Фундаментальная матрица (F, 3×3). Описывает эпиполярную геометрию между двумя видами одного 3D-объекта, снятого с разных ракурсов. Связывает точки так: x₂ᵀ F x₁ = 0. Не требует знания внутренних параметров камеры.

7 степеней свободы (det(F)=0 — одна связь, плюс нормировка).

Минимальный набор для RANSAC: 8 точек (или 7 для 7-точечного алгоритма).

Работает, когда между снимками есть смещение камеры — то есть объект не плоский, снят с разных сторон.


## Предобработка изображений

### Приведение изображений к общему виду

Набор снимков *Bosmina longirostris* обладает свойствами, которые мешают детектору работать напрямую: разное разрешение (от 778×632 до 3005×2513), смешанные цветные и чёрно-белые кадры, разная экспозиция, низкий контраст хитиновых структур. Прежде чем подавать изображения в AKAZE, приводим их к сопоставимому виду.

**Что делаем:**

1. **Перевод в grayscale.** Цветовые каналы не несут стабильной информации между снимками (разные камеры, разные режимы съёмки). Работаем с интенсивностью.

2. **Приведение к общему разрешению.** Все изображения масштабируются так, чтобы большая сторона равнялась 1200 px. Это устраняет разброс «в 4–5 раз по площади» и выравнивает работу детектора на всех снимках. Для уменьшения используется `INTER_AREA`, для увеличения — `INTER_CUBIC`.

3. **Локальное выравнивание контраста через CLAHE.** `cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))` усиливает слабоконтрастные структуры — границы створок, щетинки, антеннулы — не пережигая шум. `clipLimit=3.0` — умеренное значение, подобранное под микроскопические снимки.

4. **Повышение локальной резкости через unsharp mask.** После апскейла маленькие изображения (особенно изображение 6) выглядят размытыми. Unsharp mask возвращает высокочастотные детали: `sharp = image + amount * (image - GaussianBlur(image))`. Сила резкости зависит от исходного разрешения:
   - `amount = 1.5` для изображений с большей стороной < 1000 px (маленькое изображение 6);
   - `amount = 0.8` для остальных.

5. **Построение зеркальных версий.** AKAZE не инвариантен к отражению, а часть снимков сделана с противоположной стороны объекта. Для каждого предобработанного изображения строим горизонтально отражённую версию `images_proc_flip`. На этапе матчинга будем пробовать обе комбинации (`orig-orig` и `orig-flip`) и выбирать ту, где RANSAC находит больше inliers.

**Порядок операций в пайплайне:** `gray → resize → CLAHE → unsharp`. Резкость применяется **после** CLAHE — иначе unsharp усилит шум, который CLAHE «поднимет» вместе с полезным контрастом.

**Что получаем на выходе:**

- `images_proc` — словарь
- `images_proc_flip` — те же изображения, отражённые горизонтально;
- все изображения имеют единый масштаб и выровненный локальный контраст, что даёт AKAZE работать в сопоставимых условиях.

In [ ]:
# Функции предобработки
def to_gray(img):
    if len(img.shape) == 3:
        return cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    return img


def unsharp_mask(gray, amount=1.0, radius=3):
    """Повышение локальной резкости (unsharp mask)."""
    blurred = cv2.GaussianBlur(gray, (0, 0), radius)
    sharp = cv2.addWeighted(gray, 1 + amount, blurred, -amount, 0)
    return sharp


def apply_clahe(gray, clip=3.0, tile=8):
    clahe = cv2.createCLAHE(clipLimit=clip, tileGridSize=(tile, tile))
    return clahe.apply(gray)


def resize_to_target(img, target=1200):
    h, w = img.shape[:2]
    scale = target / max(h, w)
    interp = cv2.INTER_AREA if scale < 1 else cv2.INTER_CUBIC
    return cv2.resize(img, (int(w * scale), int(h * scale)), interpolation=interp)


def preprocess(img, target=1200):
    """gray → resize → CLAHE → unsharp.
    Резкость усиливаем сильнее для маленьких изображений."""
    gray = to_gray(img)
    h, w = gray.shape[:2]
    gray = resize_to_target(gray, target)
    gray = apply_clahe(gray, clip=3.0, tile=8)
    # сила резкости: маленькие изображения (мало пикселей) — сильнее
    amount = 1.5 if max(h, w) < 1000 else 0.8
    gray = unsharp_mask(gray, amount=amount, radius=3)
    return gray


# Строим два набора: оригинал и зеркальный
images_proc = {k: preprocess(v) for k, v in images.items()}
images_proc_flip = {k: cv2.flip(v, 1) for k, v in images_proc.items()}

print("Размеры после предобработки:")
for k, v in images_proc.items():
    print(f"  {Path(k).name}: {v.shape}")

# Визуализация: 6 рядов × 2 столбца (orig | flip)
n = len(images_proc)
fig, axes = plt.subplots(n, 2, figsize=(10, 4 * n))

for i, (k, v) in enumerate(images_proc.items()):
    # левый столбец — оригинал после предобработки
    axes[i, 0].imshow(v, cmap="gray")
    axes[i, 0].set_title(f"{Path(k).name} — orig", fontsize=10)
    axes[i, 0].axis("off")

    # правый столбец — зеркальная версия
    axes[i, 1].imshow(images_proc_flip[k], cmap="gray")
    axes[i, 1].set_title(f"{Path(k).name} — flip", fontsize=10)
    axes[i, 1].axis("off")

plt.tight_layout()
plt.show()

## Вывод по предобработке


После предобработки есть два набора изображений:

- `images_proc` — grayscale, 1200 px по большей стороне, с CLAHE и unsharp-резкостью;
- `images_proc_flip` — те же изображения, отражённые горизонтально.

Маленькое изображение 6 (778×632) после апскейла до 1200 px выглядит размытым. Unsharp mask возвращает высокочастотные детали — щетинки, края створок — делая их видимыми для AKAZE. Сила резкости адаптивна: `amount=1.5` для изображения 6, `amount=0.8` для остальных. Порядок важен: резкость применяется после CLAHE, иначе unsharp усилил бы шум, поднятый CLAHE.

AKAZE не инвариантен к отражению, а часть снимков сделана с противоположной стороны объекта. При матчинге для каждой пары `(A, B)` будем проверять обе комбинации:

- `A_orig ↔ B_orig` — обычное сопоставление;
- `A_orig ↔ B_flip` — если одно из изображений зеркально относительно другого.

Побеждает та, где RANSAC находит больше inliers. Комбинации `A_flip ↔ B_orig` и `A_flip ↔ B_flip` эквивалентны первым двум с точностью до отражения координат — их отдельно считать не нужно.




In [ ]:
# Детектор AKAZE для всех изображений
akaze = cv2.AKAZE_create(
    threshold=0.001,
    nOctaves=4,
    nOctaveLayers=4,
    diffusivity=cv2.KAZE_DIFF_PM_G2,
    descriptor_type=cv2.AKAZE_DESCRIPTOR_MLDB,
    max_points=5000
)

print("AKAZE: детектор (threshold=0.001, max_points=5000)\n")
results_proc = run_method(akaze, images_proc, "AKAZE_proc")
results_flip = run_method(akaze, images_proc_flip, "AKAZE_flip")

# фиксируем порядок путей для дальнейшей работы
paths_list = list(images_proc.keys())
N = len(paths_list)

print(f"\npaths_list создан: {N} элементов")
for i, p in enumerate(paths_list):
    print(f"  [{i}] {Path(p).name}")

# Вывод по детекции AKAZE на предобработанных изображениях

**Фактический результат:**

| Изображение | Точек (orig) | Точек (flip) |
|---|---|---|
| 1 | 5 000 | 5 000 |
| 2 | 5 000 | 5 000 |
| 3 | 5 000 | 5 000 |
| 4 | 5 000 | 5 000 |
| 5 | 5 000 | 5 000 |
| 6 | **2 567** | **2 566** |

**Что дала предобработка:**

**1. Разброс сократился с 1500× до 2×.** До предобработки на сырых изображениях было от 20 до 30 060 точек, то есть разница в 1500 раз. После предобработки — от 2 567 до 5 000, разница всего в 2 раза. Изображение 6 перестало быть выбросом: после CLAHE и unsharp mask оно даёт достаточно точек, чтобы участвовать в матчинге наравне с остальными.

**2. Изображения 1–5 упёрлись в `max_points=5000`.** Это значит, что при `threshold=0.001` AKAZE находит **больше** 5000 точек на каждом из этих изображений, а лимит обрезает «хвост» — самые слабые точки отбрасываются, остаются 5000 сильнейших. Без лимита мы бы снова получили 20–30 тысяч и вернулись к проблеме дисбаланса между изображениями.

**3. Изображение 6 даёт 2 567 точек.** Лимит не сработал — столько сильных точек там и есть. Рост с 20 (в базовом запуске без предобработки) — прямой эффект двух шагов предобработки: unsharp mask с `amount=1.5` вернул высокочастотные детали, которые AKAZE раньше не видел, а CLAHE выровнял локальный контраст, чтобы эти детали стали различимыми. Это ключевой результат предобработки: **самое слабое изображение стало полноценным участником набора**.

**4. Orig и flip дают практически идентичные числа.** 5 000 ↔ 5 000 на изображениях 1–5, 2 567 ↔ 2 566 на изображении 6. Разница в 1 точку на изображении 6 — статистический шум (лимит 5000 не сработал, число определяется порогом `threshold`, а он одинаков). Это ожидаемо: AKAZE **не инвариантен к отражению на уровне дескрипторов** (бинарные MLDB-паттерны не симметричны), но **по детекции точек он симметричен** — нелинейное масштабное пространство строится одинаково для оригинала и зеркальной версии.

**5. Время работы 0.2–0.7 с на изображение.** Суммарно 12 прогонов (6 orig + 6 flip) заняли около 4 с. Приемлемо для дальнейших экспериментов с матчингом.



На входе — сбалансированный набор из 2 500–5 000 точек на каждом изображении, в двух ориентациях (orig и flip). У каждой пары достаточно материала для устойчивого сопоставления. Все зависимости для следующего этапа готовы:

- `results_proc` — дескрипторы на предобработанных изображениях,
- `results_flip` — те же дескрипторы на зеркальных версиях,
- `paths_list` — фиксированный порядок путей (индекс 0 → изображение 1, …, индекс 5 → изображение 6),
- `N = 6` — число изображений.

Переходим к матчингу: **CrossCheck + порог по расстоянию Хэмминга + RANSAC** с мягким порогом 15 px и автовыбором модели (H или F) в зависимости от ракурса пары.

In [ ]:
def draw_keypoints_rainbow(image, keypoints, max_kp=None, radius=3):
    """Возвращает изображение с точками радужной палитрой (без показа)."""
    img_draw = image.copy()
    if len(img_draw.shape) == 2:
        img_draw = cv2.cvtColor(img_draw, cv2.COLOR_GRAY2BGR)

    if max_kp:
        keypoints = keypoints[:max_kp]

    n = len(keypoints)
    for i, kp in enumerate(keypoints):
        hue = int(180 * i / max(n, 1))
        hsv = np.uint8([[[hue, 255, 255]]])
        bgr = cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR)[0, 0].tolist()
        x, y = int(kp.pt[0]), int(kp.pt[1])
        cv2.circle(img_draw, (x, y), radius, bgr, -1)
    return img_draw


# Сетка 3 × 2, только одна версия (первые 500 точек)
n_cols = 2
n_rows = (N + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(12, 5 * n_rows))
axes = axes.ravel()

for idx, (path, (kp, des, _)) in enumerate(results_proc.items()):
    vis = draw_keypoints_rainbow(images_proc[path], kp, max_kp=500)
    axes[idx].imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
    axes[idx].set_title(f"AKAZE (первые 500): {Path(path).name}", fontsize=11)
    axes[idx].axis("off")

for k in range(len(results_proc), len(axes)):
    axes[k].axis("off")

plt.tight_layout()
plt.show()

Определяем три утилиты, которые понадобятся во всех последующих экспериментах:

match_crosscheck — BFMatcher с CrossCheck и опциональной отсечкой по расстоянию Хэмминга;

match_ratio — BFMatcher kNN(k=2) + Lowe's ratio test;

ransac_filter — геометрическая фильтрация с выбором модели H (гомография) или F (фундаментальная матрица).

Держим все параметры матчинга в одном месте — так их удобно менять и сравнивать

In [ ]:
def match_crosscheck(des1, des2, max_dist=None):
    """BFMatcher + CrossCheck. max_dist — отсечка по Хэммингу (None — без отсечки)."""
    bf = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=True)
    matches = bf.match(des1, des2)
    if max_dist is not None:
        matches = [m for m in matches if m.distance <= max_dist]
    return sorted(matches, key=lambda m: m.distance)


def match_ratio(des1, des2, ratio=0.75):
    """BFMatcher kNN(k=2) + Lowe's ratio test."""
    bf = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=False)
    knn = bf.knnMatch(des1, des2, k=2)
    good = []
    for pair in knn:
        if len(pair) < 2:
            continue
        m, n = pair
        if m.distance < ratio * n.distance:
            good.append(m)
    return sorted(good, key=lambda m: m.distance)


def ransac_filter(kp1, kp2, matches, model="H",
                  thresh=10.0, iters=5000, conf=0.999):
    """Геометрическая фильтрация. model='H' или 'F'.
    Возвращает (matches_filtered, mask, model_matrix)."""
    min_pts = 4 if model == "H" else 8
    if len(matches) < min_pts:
        return [], None, None
    src = np.float32([kp1[m.queryIdx].pt for m in matches]).reshape(-1, 1, 2)
    dst = np.float32([kp2[m.trainIdx].pt for m in matches]).reshape(-1, 1, 2)
    if model == "H":
        M, mask = cv2.findHomography(src, dst, cv2.RANSAC, thresh,
                                     maxIters=iters, confidence=conf)
    else:
        M, mask = cv2.findFundamentalMat(src, dst, cv2.FM_RANSAC, thresh,
                                         conf, maxIters=iters)
    if mask is None:
        return [], None, None
    mask = mask.ravel().astype(bool)
    return [matches[k] for k in range(len(matches)) if mask[k]], mask, M

# Подбор параметров детектора AKAZE


Перебираем параметры детектора AKAZE — порог `threshold` и лимит `max_points` — в широком диапазоне (0.0001–0.050 и 500–20 000 соответственно). Для каждой конфигурации считаем дескрипторы на паре (1, 2) с зеркальной ориентацией изображения 2 и смотрим на распределение NN-расстояний Хэмминга.

Параметры `threshold` и `max_points` определяют, **сколько и какие** точки найдёт детектор. От этого напрямую зависит всё дальнейшее: если детектор даёт мало точек — не хватит материала для сопоставления; если много слабых — матчер утонет в шуме; если у разных изображений разное число точек — возникнет дисбаланс.

Прежде чем переходить к матчингу и RANSAC, нужно понять, какая конфигурация детектора даёт **наибольший шанс** найти истинные соответствия. Минимальное расстояние между дескрипторами пары — прямой индикатор: если оно мало (20–50 бит), значит среди найденных точек есть настоящие пары; если велико (>60 бит) — детектор не находит совпадающих структур.

Диапазон подобран так, чтобы покрыть обе стороны от типичных значений:

| `threshold` | `max_points` | Что проверяем |
|---|---|---|
| 0.0001 | 20 000 | Экстремально низкий порог — максимум материала |
| 0.0005 | 10 000 | Низкий |
| 0.0010 | 5 000 | Типичное значение для слабоконтрастных снимков |
| 0.0050 | 2 000 | Умеренно выше |
| 0.0200 | 1 000 | Высокий порог |
| 0.0500 | 500 | Очень высокий — только очень сильные точки |

## Что фиксируем

- Ориентация изображения 2 — зеркальная (`flip`): по результатам предыдущего раздела известно, что изображение 2 отражено относительно 1.
- Параметры масштабного пространства AKAZE — `nOctaves=4`, `nOctaveLayers=4`, `diffusivity=KAZE_DIFF_PM_G2`, `descriptor_type=MLDB`.
- Стратегия матчинга — BFMatcher без CrossCheck и без порога: интересует чистое распределение расстояний, без фильтрации.

## Как читать результат

Для каждой конфигурации выводятся: число точек на каждом изображении, `minD` (лучшее расстояние), перцентили p1 и p5, медиана и время работы.

- **Если `minD` мало** (20–50 бит) — в наборе есть настоящие совпадения, эта конфигурация детектора перспективна.
- **Если `minD` стабильно велико** (>60 бит) при любых параметрах — детектор не находит общих структур, и причина в данных, а не в настройках.
- **Форма распределения** — одномодальное узкое означает отсутствие истинных пар, двугорбое с левым пиком — их наличие.


In [ ]:
# Поиск «рабочей зоны» AKAZE: очень низкий vs очень высокий threshold
from time import time

param_grid = [
    # (threshold, max_points) — идём в обе стороны от текущих 0.001 / 5000
    (0.0001, 20000),   # экстремально низкий порог, максимум точек
    (0.0005, 10000),   # ниже текущего
    (0.001,   5000),   # текущий
    (0.005,   2000),   # умеренно выше
    (0.020,   1000),   # высокий
    (0.050,    500),   # очень высокий
]

imgA = images_proc[paths_list[0]]
imgB = images_proc_flip[paths_list[1]]

print(f"{'thr':>9}{'max_pts':>9}{'kpA':>7}{'kpB':>7}"
      f"{'minD':>6}{'p1':>5}{'med':>6}{'p5':>5}{'t,s':>7}")
print("-" * 62)

grid_zone = {}
for thr, mp in param_grid:
    ak = cv2.AKAZE_create(
        threshold=thr, nOctaves=4, nOctaveLayers=4,
        diffusivity=cv2.KAZE_DIFF_PM_G2,
        descriptor_type=cv2.AKAZE_DESCRIPTOR_MLDB,
        max_points=mp,
    )
    t0 = time()
    kpA, desA = ak.detectAndCompute(imgA, None)
    kpB, desB = ak.detectAndCompute(imgB, None)
    if desA is None or desB is None or len(kpA) < 8 or len(kpB) < 8:
        print(f"{thr:>9.4f}{mp:>9}{len(kpA):>7}{len(kpB):>7}   -- мало точек")
        continue
    bf = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=False)
    nn = bf.match(desA, desB)
    d = np.array([m.distance for m in nn])
    dt = time() - t0
    print(f"{thr:>9.4f}{mp:>9}{len(kpA):>7}{len(kpB):>7}"
          f"{d.min():>6.0f}{np.percentile(d,1):>5.0f}"
          f"{np.median(d):>6.0f}{np.percentile(d,5):>5.0f}{dt:>7.2f}")
    grid_zone[(thr, mp)] = (kpA, desA, kpB, desB, d)

# Гистограмма для крайних случаев
keys = list(grid_zone.keys())
if len(keys) >= 2:
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    for ax, key in zip(axes, [keys[0], keys[-1]]):
        d = grid_zone[key][-1]
        ax.hist(d, bins=64, range=(0, 256), color="steelblue")
        ax.axvline(d.min(), color="r", ls="--", label=f"min={d.min():.0f}")
        ax.axvline(np.median(d), color="g", ls="--",
                   label=f"med={np.median(d):.0f}")
        ax.set_title(f"thr={key[0]}, mp={key[1]}")
        ax.set_xlabel("Hamming distance")
        ax.legend(fontsize=9)
    axes[0].set_ylabel("кол-во пар")
    plt.tight_layout()
    plt.show()

Проведён поиск оптимальных параметров детектора AKAZE по порогу threshold (0.0001–0.050) и лимиту max_points (500–20 000). На трёх самых низких порогах (0.0001, 0.0005, 0.001) минимальное расстояние Хэмминга между дескрипторами пары (1, 2) стабильно составляет 45 бит — независимо от числа найденных точек. При thr=0.0001 рассмотрено 216 миллионов парных сопоставлений (в 8.6 раз больше, чем при thr=0.001), и минимум не сдвинулся.

Что это означает. Значение 45 бит из 488 (9 % расхождения) формально попадает в диапазон, характерный для истинных совпадений MLDB-дескриптора (20–50 бит). Однако сам факт попадания в этот диапазон не является доказательством истинного совпадения: в выборке из сотен миллионов случайных пар минимальное значение неизбежно смещается в левый хвост распределения. Теоретическая оценка для нашего распределения (медиана ~111 бит, σ ≈ 16 бит) даёт ожидаемый минимум для 216M случайных пар около 40–45 бит — то есть наблюдаемое значение согласуется с гипотезой случайного совпадения, а не противоречит ей.

Признаком истинных соответствий служит не единичный минимум, а форма распределения — наличие отдельного левого пика с сотнями пар в области 20–50 бит и провала между ним и основным пиком. У нас распределение одномодальное: центр 111–122 бита, разрыв между minD=45 и первым перцентилем p1=76 составляет 31 бит, и в этом интервале в распределении практически пусто. Настоящие совпадения дали бы тысячи пар в левой части (одна структура — это десятки ключевых точек, они совпадают между двумя снимками массово), а не единичные значения. Дополнительно устойчивость minD=45 при росте выборки в 8.6 раз показывает, что новых «очень близких» пар не появляется: дальнейшее снижение порога лишь добавляет шум.

Повышение порога до 0.005 и 0.020 ухудшает минимальное расстояние (82 и 98 бит соответственно). Слабые точки, отсекаемые высоким порогом, были не хуже сильных по критерию минимального расстояния — среди них попадались лучшие кандидаты, и их удаление лишает детектор даже тех немногих относительно близких пар, которые были.

Оптимальной признана конфигурация threshold=0.001, max_points=5000. Она обеспечивает компромисс между числом точек (5000 на изображение), временем работы (~1.1 с) и качеством (minD=45, medD=122). Ни снижение порога (рост времени в 4.5 раза без улучшения minD), ни повышение (ухудшение minD) улучшений не даёт. Все дальнейшие эксперименты с AKAZE проведены на этой конфигурации.

## Подбор параметров матчинга

Проводим систематический перебор параметров попарного сопоставления на одной
репрезентативной паре изображений (1, 2). Цель — понять, при каких настройках
матчер и RANSAC начинают давать осмысленные результаты, и определить, является
ли низкое качество сопоставления следствием неудачных параметров или
принципиальным свойством данных.


**1. Стратегия матчинга дескрипторов** — 6 вариантов:

- `CrossCheck d<=100`, `d<=130`, `d<=160` — взаимная проверка ближайших
  соседей с тремя порогами по расстоянию Хэмминга. Порог задаёт «радиус
  доверия»: до 100 бит — консервативный, до 160 бит — мягкий (пропускает
  больше кандидатов, но с ростом доли мусора).
- `kNN ratio 0.80`, `0.85`, `0.90` — тест Лоу с тремя порогами. Ratio
  строже CrossCheck по построению: он требует, чтобы первый сосед был
  заметно ближе второго. Порог 0.80 — жёсткий, 0.90 — мягкий (почти как
  CrossCheck, но с сохранением k=2).

Зачем оба типа: CrossCheck даёт максимум кандидатов, ratio — максимум
«уверенности». Сравнение покажет, какой вход для RANSAC лучше на наших данных.

**2. Геометрическая модель** — 2 варианта:

- `H` — гомография, 8 степеней свободы, предполагает плоский объект или
  чистый поворот камеры.
- `F` — фундаментальная матрица, 7 степеней свободы, описывает эпиполярную
  геометрию 3D-сцены.

Зачем обе: *Bosmina* — трёхмерный объект, поэтому формально правильнее F.
Но на близких ракурсах H иногда даёт более точную подгонку. Сравнение H и F
покажет, есть ли между снимками реальный 3D-параллакс (если F заметно
выигрывает — есть).

**3. Порог перепроекции RANSAC** — 5 значений: 15, 25, 40, 60, 100 px.

- 15 px — жёсткий, соответствует «точное совпадение с учётом шума детектора».
- 25–40 px — умеренный, допускает небольшое расхождение масштаба и ракурса.
- 60–100 px — мягкий: при размере 1200 px по большей стороне это 5–8 %
  от неё, что уже сопоставимо с размером анатомической структуры
  (сложный глаз, основание антеннулы).

Зачем диапазон: если с ростом порога число inliers растёт плавно и без
насыщения — это признак случайного распределения пар: каждое расширение
окрестности вокруг модели добавляет в неё случайные точки. Наличие истинных
соответствий дало бы выход на плато — рост до определённого значения, после
которого число inliers перестаёт увеличиваться. Форма кривой
«inliers vs thr» — сама по себе диагностический признак.

### Что фиксируем

Детектор AKAZE зафиксирован на оптимальной конфигурации из предыдущего
раздела (`threshold=0.001`, `max_points=5000`). Ориентация изображения 2
зафиксирована как зеркальная (`flip`). Число итераций RANSAC
(`maxIters=5000`) и уровень доверия (`confidence=0.999`) постоянны.
Таким образом, на каждой итерации перебора меняется ровно одна переменная —
стратегия матчинга, модель или порог перепроекции, — и наблюдаемые различия
можно однозначно приписать ей.

In [ ]:
matchers = {
    "CrossCheck d<=100": lambda dA, dB: match_crosscheck(dA, dB, 100),
    "CrossCheck d<=130": lambda dA, dB: match_crosscheck(dA, dB, 130),
    "CrossCheck d<=160": lambda dA, dB: match_crosscheck(dA, dB, 160),
    "kNN ratio 0.80":    lambda dA, dB: match_ratio(dA, dB, 0.80),
    "kNN ratio 0.85":    lambda dA, dB: match_ratio(dA, dB, 0.85),
    "kNN ratio 0.90":    lambda dA, dB: match_ratio(dA, dB, 0.90),
}

thrs   = [15.0, 25.0, 40.0, 60.0, 100.0]
models = ["H", "F"]

a, b = 0, 1
kpA  = results_proc[paths_list[a]][0]
desA = results_proc[paths_list[a]][1]
kpB  = results_proc[paths_list[b]][0]
desB = results_proc[paths_list[b]][1]

print(f"{'matcher':<20}{'model':<6}{'thr':>6}"
      f"{'matches':>10}{'inliers':>10}{'%':>7}")
print("-" * 60)

grid_results = {}
for mname, mfun in matchers.items():
    m = mfun(desA, desB)
    for model in models:
        for thr in thrs:
            inl, mask, _ = ransac_filter(kpA, kpB, m, model=model, thresh=thr)
            n_inl = len(inl)
            grid_results[(mname, model, thr)] = (len(m), n_inl, inl)
            print(f"{mname:<20}{model:<6}{thr:>6.0f}"
                  f"{len(m):>10}{n_inl:>10}"
                  f"{n_inl/max(len(m),1)*100:>7.1f}")
    print()

Проведено исследование параметров попарного сопоставления на паре (1, 2) по
трём осям: стратегия матчинга (CrossCheck с max_dist = 100/130/160, kNN с
Lowe ratio 0.80/0.85/0.90), модель RANSAC (H / F) и порог перепроекции
(15–100 px). Всего рассмотрено 60 конфигураций.

Фундаментальная матрица F стабильно превосходит гомографию H по числу
inliers в 2–10 раз при одинаковых входных данных, а при малых порогах
перепроекции H может давать 0 inliers там, где F находит десятки. Это
объясняется геометрией: F накладывает на каждую пару точек одно скалярное
ограничение (лежание на эпиполярной прямой), H — два (точное совпадение
проекций). Для трёхмерного объекта (Bosmina longirostris) модель F
содержательно адекватнее.

Рост числа inliers с увеличением порога thr строго монотонный, без
насыщения. На CrossCheck d<=100 + F ряд выглядит как 27 → 36 → 50 → 66 → 90
при изменении thr с 15 до 100 px. Такая форма кривой характерна для
случайного распределения пар: каждое расширение окрестности вокруг модели
добавляет в неё случайные точки. Наличие истинных соответствий дало бы
выход на плато — он не наблюдается.

Сравнение с теоретическим ожиданием. Для случайного набора из N пар при
пороге thr ожидаемое число inliers для модели F оценивается как
N · 2·thr / W, где W — ширина изображения. При W = 1200 px, thr = 100 px
это 16.7 % от N. Реально получено 28 % (лучший случай, CrossCheck d<=160 + F)
— превышение случайного уровня лишь в 1.7 раза. Для осмысленного
сопоставления характерно превышение в 5–10 раз.

Лучшие параметры для дальнейшей работы: CrossCheck + max_dist = 100 +
F + thr = 15. Это минимально допустимый порог перепроекции, при котором
RANSAC с 7 степенями свободы даёт формально надёжные пары — 27 inliers из
242, что превышает принятый порог надёжности MIN_INLIERS = 15. Дальнейшее
ослабление порогов увеличивает абсолютное число inliers, но не улучшает
качество: доля выживших пар остаётся на уровне случайной (см. следующий
раздел — визуальный анализ линий не показывает роста семантической
согласованности).


In [ ]:
def draw_matches_color(imgA, kpA, imgB, kpB, matches,
                       max_matches=60, point_radius=5, figsize=(16, 7)):
    """Кастомная визуализация: линии разными цветами + видимые точки."""
    hA, wA = imgA.shape[:2]
    hB, wB = imgB.shape[:2]
    H = max(hA, hB)
    W = wA + wB

    # серый холст
    canvas = np.zeros((H, W, 3), dtype=np.uint8)
    canvas[:hA, :wA] = cv2.cvtColor(imgA, cv2.COLOR_GRAY2BGR)
    canvas[:hB, wA:wA + wB] = cv2.cvtColor(imgB, cv2.COLOR_GRAY2BGR)

    n = min(len(matches), max_matches)
    for idx, mm in enumerate(matches[:n]):
        # цвет из радуги: уникальный для каждой пары
        hue = int(180 * idx / max(n, 1))
        hsv = np.uint8([[[hue, 255, 255]]])
        bgr = tuple(int(c) for c in cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR)[0, 0])

        xA, yA = int(kpA[mm.queryIdx].pt[0]), int(kpA[mm.queryIdx].pt[1])
        xB, yB = int(kpB[mm.trainIdx].pt[0]), int(kpB[mm.trainIdx].pt[1])

        # точка на левом изображении
        cv2.circle(canvas, (xA, yA), point_radius, bgr, -1)
        cv2.circle(canvas, (xA, yA), point_radius + 1, (255, 255, 255), 1)

        # точка на правом изображении (со сдвигом wA)
        xB_shift = xB + wA
        cv2.circle(canvas, (xB_shift, yB), point_radius, bgr, -1)
        cv2.circle(canvas, (xB_shift, yB), point_radius + 1, (255, 255, 255), 1)

        # линия между ними
        cv2.line(canvas, (xA, yA), (xB_shift, yB), bgr, 2, cv2.LINE_AA)

        # номер пары — у левой точки
        cv2.putText(canvas, str(idx + 1), (xA + 8, yA - 8),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, bgr, 1, cv2.LINE_AA)

    plt.figure(figsize=figsize)
    plt.imshow(cv2.cvtColor(canvas, cv2.COLOR_BGR2RGB))
    plt.axis("off")
    return canvas


a, b = 0, 1
kpA = results_proc[paths_list[a]][0]
desA = results_proc[paths_list[a]][1]
kpB = results_flip[paths_list[b]][0]
desB = results_flip[paths_list[b]][1]

m = match_crosscheck(desA, desB, max_dist=100)

configs = [
    ("H, thr=15 (baseline)", "H",  15.0),
    ("F, thr=15 (честная)",  "F",  15.0),
    ("F, thr=100 (мягкая)",  "F", 100.0),
]

for title, model, thr in configs:
    inl, mask, _ = ransac_filter(kpA, kpB, m, model=model, thresh=thr)
    print(f"\n{title} (flip img2) — inliers: {len(inl)} / {len(m)}")
    canvas = draw_matches_color(
        images_proc[paths_list[a]], kpA,
        images_proc_flip[paths_list[b]], kpB,
        inl, max_matches=60,
    )
    plt.title(f"{title} (flip img2) — inliers: {len(inl)} / {len(m)}",
              fontsize=12)
    plt.show()

Визуализация RANSAC-соответствий с уникальными цветами линий и
пронумерованными точками показала, что **лишь незначительное меньшинство
связей** согласуется с реальными анатомическими структурами. Даже в наиболее
строгой конфигурации (F, thr=15) из 27 найденных inliers только единичные
линии совпадают с контурами створки, основанием антеннул и границами
сложного глаза. Подавляющее большинство линий при этом связывает точки,
не имеющие общего анатомического смысла, — визуально они либо пересекаются
с другими, либо соединяют различающиеся структуры. Связной картины
соответствий не образуется.

С ростом порога (F, thr=100) абсолютное число inliers увеличивается почти
в 3.5 раза (с 27 до 90), однако **доля корректных связей при этом падает,
а не растёт**. Каждое ослабление порога добавляет в набор преимущественно
ложные соответствия: линии начинают соединять сложный глаз с задним краем
створки, антеннулу с мукро, основание постабдомена с передним краем.
Перекрестий становится больше, семантической осмысленности — меньше. Это
означает, что RANSAC с ростом порога не «находит больше правильных пар»,
а просто расширяет окрестность вокруг модели, втягивая в неё случайные
точки.

**Ключевое наблюдение.** Даже на отдельно взятой паре (1, 2) — самой
«дружественной» из всех 15 по числу CrossCheck-кандидатов — доля
семантически корректных связей оценивается в единицы процентов от числа
inliers. При переходе к другим парам картина не улучшается, а ухудшается:
изображения сняты в разных ракурсах, с разной экспозицией, часть —
зеркально. На остальных 14 парах наблюдается та же картина — единичные
правильные связи на фоне хаотичных перекрестий, при этом ни одна пара
не даёт связного набора соответствий.

**Совокупный вывод.** Набор изображений **не подходит для классических
методов поиска локальных особенностей**. Даже на уровне попарного
сопоставления — где задача наиболее проста — методы дают лишь единичные
корректные соответствия, не образуя устойчивой структуры. Общие ключевые
точки, согласованные одновременно на всех шести изображениях, в терминах
SIFT / ORB / AKAZE / BRISK **не существуют**. Это не результат неудачных
параметров, а следствие природы данных: сравниваются разные особи одного
вида, у которых гомологичные структуры не обладают общей пиксельной
текстурой, необходимой для работы дескрипторов локальных особенностей.

Для независимого подтверждения вывода проведён анализ распределений
расстояний между дескрипторами. Этот анализ не зависит от выбора модели
RANSAC и стратегии матчинга и позволяет установить, содержится ли в наборе
дескрипторов статистически различимая группа истинных соответствий.
Результаты приведены в следующем разделе.

Строим распределения NN-расстояний Хэмминга и Lowe ratio между дескрипторами
AKAZE для пары (1, 2) в четырёх режимах: raw_orig, raw_flip, prep_orig,
prep_flip. Для каждого режима выводим числовые характеристики распределения
и две гистограммы — d1 и d1/d2.

Предыдущие разделы (RANSAC, визуализация линий) дали косвенные аргументы:
они зависят от выбора модели, порога перепроекции и стратегии матчинга.
Распределения расстояний — прямой аргумент, не зависящий от параметров
фильтрации.

Если в данных есть истинные соответствия, они дадут отдельный левый пик в
распределении d1 (несколько сотен пар в области 20–50 бит) с провалом между
ним и основным пиком, а также левую асимметрию (среднее < медианы). Ratio
при этом сдвинется к 0.5–0.7 с пиком в этой области. Если соответствий нет
— распределения будут одномодальными, симметричными, узкими, без левого
пика, с ratio, центрированным около 0.95 (среднее и медиана близки).

### Что варьируем

2 × 2: предобработка (raw vs CLAHE + unsharp) × зеркальность (orig vs flip
изображения 2). Матрица позволяет разделить влияние этих факторов — при
одной конфигурации было бы неясно, что именно определяет форму.

Каждый из четырёх режимов проверяет свою гипотезу:
- `raw_orig` — базовая линия: сырое grayscale без обработки, без зеркала.
  Показывает, что даёт детектор на «чистых» данных.
- `raw_flip` — то же с зеркальным изображением 2. Проверяет, улучшает ли
  зеркальность распределение (известно, что изображение 2 отражено
  относительно 1).
- `prep_orig` — предобработка без зеркала. Проверяет, помогает ли
  CLAHE + unsharp выделить устойчивые дескрипторы.
- `prep_flip` — предобработка + зеркало. Комбинация обоих эффектов.

### Параметры

Детектор AKAZE — как в основном пайплайне (`threshold=0.001`,
`max_points=5000`). Параметры попарного сопоставления (`CrossCheck d<=100`,
`F`, `thr=15`) не применяются: раздел исследует сырые расстояния без
фильтрации. Порог по расстоянию не вводится намеренно — форма полного
распределения несёт информацию о наличии истинных пар.


In [ ]:
# Полный набор: 4 варианта пары (1,2)

paths_list = list(images_proc.keys())
i, j = 0, 1


def raw_gray(p):
    return cv2.cvtColor(images[p], cv2.COLOR_BGR2GRAY)


ak_probe = cv2.AKAZE_create(
    threshold=0.001, nOctaves=4, nOctaveLayers=4,
    diffusivity=cv2.KAZE_DIFF_PM_G2,
    descriptor_type=cv2.AKAZE_DESCRIPTOR_MLDB,
    max_points=5000
)

raw_gray_i = raw_gray(paths_list[i])
raw_gray_j = raw_gray(paths_list[j])

variants = {
    "raw_orig":  (raw_gray_i, raw_gray_j),
    "raw_flip":  (raw_gray_i, cv2.flip(raw_gray_j, 1)),
    "prep_orig": (images_proc[paths_list[i]], images_proc[paths_list[j]]),
    "prep_flip": (images_proc[paths_list[i]], images_proc_flip[paths_list[j]]),
}

bf_nn  = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=False)
bf_knn = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=False)

stats = {}
for name, (imgA, imgB) in variants.items():
    kpA, desA = ak_probe.detectAndCompute(imgA, None)
    kpB, desB = ak_probe.detectAndCompute(imgB, None)

    nn  = bf_nn.match(desA, desB)
    knn = bf_knn.knnMatch(desA, desB, k=2)

    d1_all = np.array([m.distance for m in nn])
    d1 = np.array([p[0].distance for p in knn])
    d2 = np.array([p[1].distance for p in knn])
    ratio = d1 / np.maximum(d2, 1e-6)

    stats[name] = dict(
        kpA=kpA, desA=desA, kpB=kpB, desB=desB,
        imgA=imgA, imgB=imgB,
        d1_all=d1_all, ratio=ratio,
    )

order = ["raw_orig", "raw_flip", "prep_orig", "prep_flip"]

# текстовая сводка

print("СРАВНЕНИЕ 4 ВАРИАНТОВ ПАРЫ (1,2): raw/prep × orig/flip\n")


for name in order:
    s = stats[name]
    d1_all = s["d1_all"]
    ratio  = s["ratio"]

    p_d1 = {q: float(np.percentile(d1_all, q))
            for q in (1, 5, 25, 50, 75, 95, 99)}
    p_r  = {q: float(np.percentile(ratio, q))
            for q in (1, 5, 25, 50, 75, 95, 99)}

    frac_d1_50  = float((d1_all < 50).mean())
    frac_d1_80  = float((d1_all < 80).mean())
    frac_d1_100 = float((d1_all < 100).mean())

    frac_070 = float((ratio < 0.70).mean())
    frac_075 = float((ratio < 0.75).mean())
    frac_080 = float((ratio < 0.80).mean())
    frac_085 = float((ratio < 0.85).mean())

    gap = float(np.median(d1_all) - d1_all.min())

    print(f"ВАРИАНТ: {name}\n")
    print(f"  Ключевых точек: Img1 = {len(s['kpA'])}, Img2 = {len(s['kpB'])}")
    print(f"  Размер дескриптора: {s['desA'].shape[1]} байт = "
          f"{s['desA'].shape[1] * 8} бит")
    print(f"  Число NN-пар: {len(d1_all)}")

    print(f"\n  NN-расстояния d1:")
    print(f"    min={d1_all.min():.0f}  max={d1_all.max():.0f}  "
          f"mean={d1_all.mean():.2f}  std={d1_all.std():.2f}")
    print(f"    перцентили: " + "  ".join(
        f"p{q}={v:.0f}" for q, v in p_d1.items()))
    print(f"    доля d1<50: {frac_d1_50:.2%} | d1<80: {frac_d1_80:.2%} | "
          f"d1<100: {frac_d1_100:.2%}")

    print(f"\n  Lowe ratio (d1/d2):")
    print(f"    min={ratio.min():.3f}  max={ratio.max():.3f}  "
          f"mean={ratio.mean():.3f}  std={ratio.std():.3f}")
    print(f"    перцентили: " + "  ".join(
        f"p{q}={v:.3f}" for q, v in p_r.items()))
    print(f"    доля ratio<0.70: {frac_070:.2%} | <0.75: {frac_075:.2%} | "
          f"<0.80: {frac_080:.2%} | <0.85: {frac_085:.2%}")

    print(f"\n  Форма распределения:")
    print(f"    медиана - min = {gap:.1f} бит  \n")

# графики

print("ГРАФИКИ")


fig, axes = plt.subplots(len(order), 2, figsize=(14, 5 * len(order)))
if len(order) == 1:
    axes = axes.reshape(1, -1)

for row, name in enumerate(order):
    s = stats[name]
    d1_all = s["d1_all"]
    ratio  = s["ratio"]

    ax = axes[row, 0]
    ax.hist(d1_all, bins=64, range=(0, 256), color="steelblue")
    for x, c in [(50, "r"), (80, "g"), (100, "orange")]:
        ax.axvline(x, color=c, ls="--", alpha=0.7, label=f"max_dist={x}")
    ax.axvline(d1_all.min(), color="k", ls=":", alpha=0.9,
               label=f"min={d1_all.min():.0f}")
    ax.axvline(np.median(d1_all), color="magenta", ls=":", alpha=0.9,
               label=f"med={np.median(d1_all):.0f}")
    ax.set_title(f"NN-расстояние Хэмминга | {name}", fontsize=11)
    ax.set_xlabel("Hamming distance"); ax.set_ylabel("кол-во пар")
    ax.legend(fontsize=8)

    ax = axes[row, 1]
    ax.hist(np.clip(ratio, 0, 1), bins=64, range=(0, 1), color="darkorange")
    for r in (0.7, 0.75, 0.8):
        ax.axvline(r, color="k", ls="--", alpha=0.6, label=f"ratio={r}")
    frac_075 = float((ratio < 0.75).mean())
    frac_080 = float((ratio < 0.80).mean())
    ax.set_title(f"Lowe ratio | {name} | "
                 f"<0.75: {frac_075:.1%}  <0.80: {frac_080:.1%}", fontsize=11)
    ax.set_xlabel("d1 / d2"); ax.set_ylabel("кол-во пар")
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

Построены распределения NN-расстояний Хэмминга и Lowe ratio между
дескрипторами AKAZE для пары (1, 2) в четырёх режимах: raw_orig, raw_flip,
prep_orig, prep_flip. Во всех режимах распределения одномодальные,
симметричные, узкие: медиана 118–122 бита, стандартное отклонение 14–16 бит,
диапазон p5–p95 — 90–145 бит. Средние значения совпадают с медианами с
точностью до 0.5 бита, что указывает на отсутствие левой асимметрии и,
следовательно, статистически различимой группы пар с малым расстоянием —
любая такая группа (даже несколько десятков пар) сдвинула бы среднее влево
от медианы.

Минимальное расстояние составляет 45–67 бит из 488 (9–14 %), при этом
разрыв между медианой и минимумом (55–77 бит) плавный, без отдельной группы.
Для истинного соответствия ожидается группа пар с расстоянием 20–40 бит
(несколько сотен из 5000), создающая отдельный левый пик, а также разрыв
80–100 бит между медианой и минимумом этой группы. Доля пар с d1 < 80 —
0.12–1.12 %, доля с ratio < 0.75 — 0.20–0.48 %. Средние значения Lowe ratio
(0.948–0.951) указывают на отсутствие дискриминирующей способности
дескрипторов: первый и второй ближайшие соседи почти неразличимы.

Зеркальность снижает минимальное расстояние на 9–22 бита, но не изменяет
медиану и форму распределения. Предобработка CLAHE + unsharp mask сдвигает
медиану вправо (118 → 122) и ухудшает минимум в orig-варианте (54 → 67),
что указывает на добавление высокочастотной составляющей без устойчивой
информации. Ни один из факторов не приводит к появлению левого пика.

Заключение. Распределения дескрипторов статистически несовместимы с
гипотезой о наличии истинных соответствий. Совместно с результатами
попарного сопоставления (RANSAC-inliers, визуализация линий) это
подтверждает: классические методы поиска локальных особенностей
(SIFT, ORB, AKAZE, BRISK) принципиально неприменимы к данному набору,
поскольку сравниваются разные особи одного вида, у которых гомологичные
структуры не обладают общей пиксельной текстурой. Общие ключевые точки,
согласованные на всех шести изображениях, в терминах этих методов не
существуют. Решение задачи требует семантического подхода — обученных
нейросетевых моделей или ручной разметки морфологических ориентиров.



## Нейросетевой подход: LoFTR

### Почему переходим к нейросетям

Классические методы поиска локальных особенностей (SIFT, ORB, AKAZE, BRISK)
показали неудовлетворительный результат на нашем наборе. Распределения
NN-расстояний одномодальные без левого пика, доля пар с `ratio < 0.75` не
превышает 0.5 %, при попарном RANSAC-сопоставлении доля семантически
корректных связей — единицы процентов. Причина — различие пиксельной
текстуры гомологичных структур у разных особей *Bosmina longirostris*.

В этом разделе проверяем **нейросетевой подход**: обученный dense-матчер
**LoFTR** (Local Feature TRansformer). В отличие от классических методов,
LoFTR не ищет повторяющиеся пиксельные паттерны — он использует
**attention-механизм** на грубом разрешении, сопоставляя изображения на
уровне **семантических признаков**. Это принципиально другой подход:
вместо «одинаковых пикселей» — «похожие по смыслу участки».

### Что такое LoFTR

LoFTR — transformer-based метод (Sun et al., 2021). Архитектурно:

- **Backbone (ResNet + FPN):** многомасштабные признаки с пониженным
  разрешением (1/8 от исходного).
- **Transformer (self + cross attention):** каждая точка одного изображения
  «общается» со всеми точками другого через cross-attention — это позволяет
  сопоставлять участки с **глобальным контекстом**, а не только по локальной
  текстуре.
- **Coarse matching:** на грубом уровне (1/8) строится матрица соответствий
  между всеми пикселями.
- **Fine matching:** найденные соответствия уточняются до пиксельного
  разрешения.

Модель обучена на больших корпусах (MegaDepth, ScanNet) и не требует
дообучения на наших данных.

### Преимущества LoFTR для нашей задачи

| Преимущество | Почему важно здесь |
|---|---|
| Semantic matching | Сопоставляет структуры по форме и контексту, а не по пиксельной текстуре |
| Dense correspondences | Даёт сотни матчей на пару, в отличие от AKAZE с 0–3 корректными |
| Confidence score | Каждый матч имеет оценку уверенности для фильтрации |
| Не требует детектора | Работает end-to-end, не зависит от качества ключевых точек |
| Устойчивость к освещению | Обучен на данных с разной экспозицией |

### Что настраиваем

- **Модель:** `LoFTR(pretrained='outdoor')` — для наружных сцен, лучше
  подходит для микроскопии, чем `indoor`.
- **Порог `confidence`:** 0.4 — мягкий, сохраняет больше кандидатов;
  RANSAC потом отсеет случайные.
- **Ориентация:** фиксированные флаги из предыдущего анализа
  (`[F, T, T, F, T, F]` — изображения 2, 3, 5 отражены).

In [ ]:
# Устанавливаем нужные библиотеки
!pip install -q kornia kornia-rs
import kornia as K
import kornia.feature as KF
import torch
import cv2
import numpy as np
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Используем устройство: {device}")

matcher = KF.LoFTR(pretrained='outdoor').to(device)


def match_loftr(img1_gray, img2_gray):
    img1 = torch.from_numpy(img1_gray).float()[None, None] / 255.0
    img2 = torch.from_numpy(img2_gray).float()[None, None] / 255.0
    img1, img2 = img1.to(device), img2.to(device)
    with torch.inference_mode():
        correspondences = matcher({'image0': img1, 'image1': img2})
    kp1 = correspondences['keypoints0'].cpu().numpy()
    kp2 = correspondences['keypoints1'].cpu().numpy()
    conf = correspondences['confidence'].cpu().numpy()
    return kp1, kp2, conf


def draw_dense(imgA, kpA, imgB, kpB, conf, title="", max_show=80):
    hA, wA = imgA.shape[:2]
    hB, wB = imgB.shape[:2]
    canvas = np.zeros((max(hA, hB), wA + wB, 3), dtype=np.uint8)
    canvas[:hA, :wA] = cv2.cvtColor(imgA, cv2.COLOR_GRAY2BGR)
    canvas[:hB, wA:wA + wB] = cv2.cvtColor(imgB, cv2.COLOR_GRAY2BGR)

    order = np.argsort(-conf)
    kpA, kpB, conf = kpA[order], kpB[order], conf[order]

    n = min(len(kpA), max_show)
    for idx in range(n):
        hue = int(180 * idx / max(n, 1))
        bgr = tuple(int(c) for c in cv2.cvtColor(
            np.uint8([[[hue, 255, 255]]]), cv2.COLOR_HSV2BGR)[0, 0])
        xA, yA = int(kpA[idx][0]), int(kpA[idx][1])
        xB, yB = int(kpB[idx][0]), int(kpB[idx][1])

        # тонкая линия: 1 px, только для очень уверенных 2 px
        w_line = 1 if conf[idx] < 0.85 else 2

        # маленькие точки без обводки
        cv2.circle(canvas, (xA, yA), 3, bgr, -1)
        cv2.circle(canvas, (xB + wA, yB), 3, bgr, -1)
        cv2.line(canvas, (xA, yA), (xB + wA, yB), bgr, w_line, cv2.LINE_AA)

    plt.figure(figsize=(16, 7))
    plt.imshow(cv2.cvtColor(canvas, cv2.COLOR_BGR2RGB))
    plt.title(title, fontsize=12)
    plt.axis('off')
    plt.tight_layout()
    plt.show()


# === Три лучшие пары ===
TOP_PAIRS = [
    (3, 4, "flip"),
    (0, 1, "flip"),
    (1, 3, "flip"),
]

CONF_THR = 0.4   # ← порог поднят с 0.5 до 0.6

for (a, b, orient) in TOP_PAIRS:
    img1_gray = images_proc[paths_list[a]]
    img2_gray = (images_proc[paths_list[b]] if orient == "orig"
                 else images_proc_flip[paths_list[b]])

    kp1, kp2, conf = match_loftr(img1_gray, img2_gray)
    keep = conf > CONF_THR
    kp1_f, kp2_f, conf_f = kp1[keep], kp2[keep], conf[keep]

    print(f"\nПара ({a+1},{b+1}) {orient}:")
    print(f"  raw: {len(kp1)}  |  conf>{CONF_THR}: {len(kp1_f)}")
    if len(conf_f) > 0:
        print(f"  conf: min={conf_f.min():.3f}  med={np.median(conf_f):.3f}  "
              f"max={conf_f.max():.3f}")

    draw_dense(img1_gray, kp1_f, img2_gray, kp2_f, conf_f,
               title=f"LoFTR ({a+1},{b+1}) {orient} — {len(kp1_f)} matches "
                     f"(conf > {CONF_THR})")

## Результаты LoFTR

### Количественные результаты

| Пара | Raw матчей | conf > 0.4 | med conf | max conf |
|---|---|---|---|---|
| (4,5) flip | 537 | **155** | 0.530 | 0.969 |
| (1,2) flip | 624 | **160** | 0.498 | 0.933 |
| (2,4) flip | 416 | **75** | 0.488 | 0.865 |

Для сравнения: AKAZE с RANSAC на тех же парах давал **0–3 корректные
связи** из 200–500 CrossCheck-кандидатов. LoFTR даёт **75–160** устойчивых
матчей после порога — в **50–100 раз больше**.

### Что это означает

LoFTR нашёл соответствия там, где классические
методы не справлялись. Сырых матчей 416–624, после фильтрации 75–160 —
это статистически значимая структура, с которой можно работать.

**Уверенность матчей.** Медиана confidence 0.49–0.53 — примерно половина
матчей имеет уверенность выше среднего, максимумы достигают 0.86–0.97.
Среди найденных связей есть действительно уверенные. У AKAZE такой шкалы
уверенности нет вообще.

**Визуально — линии в основном параллельны.** Большинство связей идёт в
одном направлении, что означает: LoFTR нашёл **глобальное согласованное
преобразование** между изображениями. Это принципиально лучше «веера»
от AKAZE, где связи пересекались под разными углами.

**Промахи есть, но они в меньшинстве.** Часть линий соединяет разные
структуры — это ожидаемо для метода, работающего на грубом разрешении
(1/8 пикселя). Такие промахи — не признак неработоспособности, а шум,
который фильтруется порогом `confidence > 0.5` и RANSAC-фильтрацией.

### Сравнение с AKAZE

| Критерий | AKAZE | LoFTR |
|---|---|---|
| Матчей после фильтрации | 0–3 корректных | 75–160 |
| Направление линий | «Веер», случайные углы | Преимущественно параллельные |
| Уверенность | Нет оценки | 0.4–0.97 |
| Устойчивость к разным особям | Низкая | **Существенно выше** |

### Итог

**LoFTR справляется с задачей качественно лучше классических методов**
и даёт осмысленный набор соответствий даже на изображениях разных особей.
Это подтверждает гипотезу: проблема классических методов — не в природе
задачи, а в их принципе работы (пиксельные паттерны вместо семантических
признаков). LoFTR обходит это ограничение через attention-механизм и
находит структуры, гомологичные по форме.

**Следующий шаг** — использовать LoFTR-матчи для построения графа общих
ключевых точек на всех 6 изображениях: кластеризация, Union-Find,
верификация через прямые связи. Результаты приведены в следующем разделе.

## Построение графа общих точек через LoFTR

После того как LoFTR показал существенно больше матчей, чем классические
методы (75–160 устойчивых связей на пару против 0–3 у AKAZE), переходим к
построению **графа общих ключевых точек** на всех 6 изображениях.

### Стратегия

Задача «найти точки, общие для всех 6 изображений» разбивается на
последовательность шагов:

1. **Попарный матчинг LoFTR** на всех C(6,2) = 15 парах с фиксированными
   ориентациями изображений (2, 3, 5 отражены — установлено ранее).
2. **RANSAC-фильтрация** (фундаментальная матрица, порог 15 px) для удаления
   геометрически несогласованных матчей.
3. **Кластеризация точек** внутри каждого изображения через DBSCAN
   (`eps=20 px`): точки, попавшие в один кластер, считаются одной физической
   структурой.
4. **Построение графа соответствий**: узлы — `(image_id, cluster_id)`,
   рёбра — RANSAC-фильтрованные LoFTR-матчи.
5. **Union-Find** для объединения узлов в **треки** — связные компоненты,
   соответствующие одной и той же структуре на разных изображениях.
6. **Верификация**: трек считается валидным, только если **для каждой пары
   изображений** внутри трека есть **прямое ребро** (не цепочка через другие
   изображения). Это защита от «дрейфа» — ситуации, когда A связан с B,
   B связан с C, но A и C не связаны.
7. **Визуализация** топ-30 треков в канонических ориентациях с одинаковыми
   номерами.

### Почему именно такая схема

Классические методы решают задачу **попарно**, они не умеют искать точки
сразу на N изображениях. Граф + Union-Find — стандартный приём в
Structure-from-Motion (feature tracks): он естественно обобщает попарные
матчи на произвольное число изображений и позволяет проверить
**согласованность** связи через все пары, а не только через цепочку.

In [ ]:
# === LoFTR с фиксированными ориентациями (2, 3, 5 — flip) ===
import torch
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from collections import defaultdict, Counter
from sklearn.cluster import DBSCAN
from kornia.feature import LoFTR

# --- Восстановление, если runtime сбрасывался ---
if "paths_list" not in globals() or "images_proc" not in globals():
    DATA_DIR = "images"
    for p in Path(DATA_DIR).iterdir():
        if p.is_file() and p.suffix == "":
            p.rename(p.with_suffix(".jpg"))
    paths = sorted(Path(DATA_DIR).rglob("*.jpg"))
    images = {str(p): cv2.imread(str(p)) for p in paths}

    def to_gray(img):
        return cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if img.ndim == 3 else img
    def unsharp_mask(g, amount=1.0, radius=3):
        b = cv2.GaussianBlur(g, (0, 0), radius)
        return cv2.addWeighted(g, 1 + amount, b, -amount, 0)
    def apply_clahe(g, clip=3.0, tile=8):
        return cv2.createCLAHE(clipLimit=clip, tileGridSize=(tile, tile)).apply(g)
    def resize_to_target(img, target=1200):
        h, w = img.shape[:2]
        s = target / max(h, w)
        interp = cv2.INTER_AREA if s < 1 else cv2.INTER_CUBIC
        return cv2.resize(img, (int(w * s), int(h * s)), interpolation=interp)
    def preprocess(img, target=1200):
        gray = to_gray(img)
        h, w = gray.shape[:2]
        gray = resize_to_target(gray, target)
        gray = apply_clahe(gray)
        amount = 1.5 if max(h, w) < 1000 else 0.8
        return unsharp_mask(gray, amount=amount, radius=3)

    images_proc = {k: preprocess(v) for k, v in images.items()}
    paths_list = list(images_proc.keys())

N = len(paths_list)

# === ФИКСИРОВАННЫЕ ОРИЕНТАЦИИ ===
# True = изображение отражено (flip), False = как есть (orig)
FLIP_FLAGS = [False, True, True, False, True, False]   # 1,2,3,4,5,6

# Канонические изображения для каждого индекса
canonical_imgs = [
    cv2.flip(images_proc[paths_list[i]], 1) if FLIP_FLAGS[i] else images_proc[paths_list[i]]
    for i in range(N)
]
print("Канонические ориентации:")
for i in range(N):
    print(f"  Img{i+1}: {'FLIP' if FLIP_FLAGS[i] else 'orig'}")


# --- LoFTR ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\nDevice: {device}")
matcher = LoFTR(pretrained='outdoor').to(device).eval()

def match_loftr(imgA, imgB):
    tA = torch.from_numpy(imgA).float()[None, None] / 255.0
    tB = torch.from_numpy(imgB).float()[None, None] / 255.0
    tA, tB = tA.to(device), tB.to(device)
    with torch.inference_mode():
        corr = matcher({'image0': tA, 'image1': tB})
    return (corr['keypoints0'].cpu().numpy(),
            corr['keypoints1'].cpu().numpy(),
            corr['confidence'].cpu().numpy())


# === ШАГ 1. LoFTR с фиксированными ориентациями ===
CONF_THR = 0.4
RANSAC_THR = 15.0

pair_best = {}
print(f"\n{'пара':<8}{'rel':<7}{'conf>0.4':>10}{'RANSAC':>10}")
print("-" * 36)

for a in range(N):
    for b in range(a + 1, N):
        # относительная ориентация: если флаги совпадают — orig, иначе — flip
        rel = "orig" if FLIP_FLAGS[a] == FLIP_FLAGS[b] else "flip"
        imgA = canonical_imgs[a]
        imgB = canonical_imgs[b]
        if rel == "flip":
            imgB = cv2.flip(imgB, 1)

        kpA, kpB, conf = match_loftr(imgA, imgB)
        keep = conf > CONF_THR
        kpA_f, kpB_f, conf_f = kpA[keep], kpB[keep], conf[keep]

        if len(kpA_f) < 8:
            pair_best[(a, b)] = dict(n_conf=len(kpA_f), n_inl=0,
                                     kpA=None, kpB=None, conf=None)
            print(f"({a+1},{b+1})  {rel:<7}{len(kpA_f):>10}{0:>10}  -- мало")
            continue

        src = np.float32(kpA_f).reshape(-1, 1, 2)
        dst = np.float32(kpB_f).reshape(-1, 1, 2)
        F, mask = cv2.findFundamentalMat(
            src, dst, cv2.FM_RANSAC, RANSAC_THR, 0.999, maxIters=5000)

        if mask is None:
            pair_best[(a, b)] = dict(n_conf=len(kpA_f), n_inl=0,
                                     kpA=None, kpB=None, conf=None)
            print(f"({a+1},{b+1})  {rel:<7}{len(kpA_f):>10}{0:>10}  RANSAC fail")
            continue

        mask = mask.ravel().astype(bool)
        pair_best[(a, b)] = dict(
            n_conf=len(kpA_f), n_inl=int(mask.sum()),
            kpA=kpA_f[mask], kpB=kpB_f[mask], conf=conf_f[mask],
        )
        print(f"({a+1},{b+1})  {rel:<7}{len(kpA_f):>10}{mask.sum():>10}")

tot_inl = sum(p["n_inl"] for p in pair_best.values())
print(f"\nВсего RANSAC-inliers: {tot_inl}")


# === ШАГ 2. Кластеризация ===
CLUSTER_EPS = 20

points_per_image = {i: [] for i in range(N)}
for (a, b), res in pair_best.items():
    if res["kpA"] is None:
        continue
    for (x, y) in res["kpA"]:
        points_per_image[a].append((x, y))
    for (x, y) in res["kpB"]:
        points_per_image[b].append((x, y))

image_clusters = {}
for i in range(N):
    pts = points_per_image[i]
    if not pts:
        image_clusters[i] = dict(labels=np.array([]), coords=np.array([]))
        continue
    coords = np.array(pts)
    labels = DBSCAN(eps=CLUSTER_EPS, min_samples=1).fit_predict(coords)
    image_clusters[i] = dict(labels=labels, coords=coords)

print("\nКластеров на изображение:")
for i in range(N):
    n_pts = len(image_clusters[i]["labels"])
    n_cl = len(set(image_clusters[i]["labels"])) if n_pts else 0
    print(f"  Img{i+1}: {n_pts} точек → {n_cl} кластеров")


def get_cluster_id(img_id, x, y):
    coords = image_clusters[img_id]["coords"]
    labels = image_clusters[img_id]["labels"]
    if len(coords) == 0:
        return -1
    d = np.hypot(coords[:, 0] - x, coords[:, 1] - y)
    return int(labels[int(np.argmin(d))])


# === ШАГ 3. Граф + Union-Find ===
class UnionFind:
    def __init__(self):
        self.parent = {}
    def find(self, x):
        if x not in self.parent:
            self.parent[x] = x
            return x
        root = x
        while self.parent[root] != root:
            root = self.parent[root]
        while self.parent[x] != root:
            self.parent[x], x = root, self.parent[x]
        return root
    def union(self, x, y):
        rx, ry = self.find(x), self.find(y)
        if rx != ry:
            self.parent[rx] = ry


uf = UnionFind()
for (a, b), res in pair_best.items():
    if res["kpA"] is None:
        continue
    for (x1, y1), (x2, y2) in zip(res["kpA"], res["kpB"]):
        c1 = get_cluster_id(a, x1, y1)
        c2 = get_cluster_id(b, x2, y2)
        if c1 < 0 or c2 < 0:
            continue
        uf.union((a, c1), (b, c2))

tracks = defaultdict(set)
for node in list(uf.parent.keys()):
    tracks[uf.find(node)].add(node)

print(f"\nВсего узлов: {len(uf.parent)}")
print(f"Всего треков: {len(tracks)}")

cover_counter = Counter(len({img for img, _ in tr}) for tr in tracks.values())
print("Покрытие изображений:")
for k in sorted(cover_counter):
    print(f"  {k} изобр.: {cover_counter[k]} треков")

full_tracks = [tr for tr in tracks.values() if len({i for i, _ in tr}) == N]
print(f"\nТреков на все {N} изображений: {len(full_tracks)}")


# === ШАГ 4. Верификация (прямое ребро для каждой пары) ===
def has_direct_link(ii, ci, jj, cj):
    key = (min(ii, jj), max(ii, jj))
    if key not in pair_best or pair_best[key]["kpA"] is None:
        return False
    res = pair_best[key]
    ptsA = res["kpA"] if ii < jj else res["kpB"]
    ptsB = res["kpB"] if ii < jj else res["kpA"]
    for (x1, y1), (x2, y2) in zip(ptsA, ptsB):
        if get_cluster_id(ii, x1, y1) == ci and get_cluster_id(jj, x2, y2) == cj:
            return True
    return False


def verify_track(track):
    imgs = sorted({i for i, _ in track})
    cmap = {i: c for i, c in track}
    for k in range(len(imgs)):
        for m in range(k + 1, len(imgs)):
            i, j = imgs[k], imgs[m]
            if not has_direct_link(i, cmap[i], j, cmap[j]):
                return False
    return True


verified = [tr for tr in full_tracks if verify_track(tr)]
print(f"Треков с прямыми связями для всех пар: {len(verified)}")


# === ШАГ 5. Визуализация в КАНОНИЧЕСКИХ ориентациях ===
if verified:
    tracks_to_show = verified[:30]
    title_suffix = f"общих точек: {len(tracks_to_show)}"
else:
    partial = sorted(tracks.values(),
                     key=lambda tr: len({i for i, _ in tr}), reverse=True)
    tracks_to_show = [tr for tr in partial if len({i for i, _ in tr}) >= 4][:30]
    title_suffix = f"треков (макс. покрытие)"
    print(f"\n⚠ Полных треков нет. Показаны треки с покрытием ≥4 из 6 "
          f"(всего {len(tracks_to_show)}).")

fig, axes = plt.subplots(2, 3, figsize=(22, 12))
axes = axes.ravel()

for idx in range(N):
    img = canonical_imgs[idx].copy()
    if img.ndim == 2:
        img = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)

    n_shown = 0
    for k, tr in enumerate(tracks_to_show, 1):
        node = next((n for n in tr if n[0] == idx), None)
        if node is None:
            continue
        _, cluster_id = node
        coords = image_clusters[idx]["coords"]
        labels = image_clusters[idx]["labels"]
        cluster_pts = coords[labels == cluster_id]
        if len(cluster_pts) == 0:
            continue
        cx, cy = cluster_pts.mean(axis=0)

        # круг побольше
        cv2.circle(img, (int(cx), int(cy)), 22, (0, 255, 0), 3)

        # номер — большой, с чёрной обводкой для читаемости
        text = str(k)
        pos = (int(cx) + 26, int(cy) - 22)
        # чёрная обводка
        cv2.putText(img, text, pos,
                    cv2.FONT_HERSHEY_SIMPLEX, 1.8, (0, 0, 0), 7, cv2.LINE_AA)
        # жёлтый текст сверху
        cv2.putText(img, text, pos,
                    cv2.FONT_HERSHEY_SIMPLEX, 1.8, (0, 255, 255), 4, cv2.LINE_AA)
        n_shown += 1

    orient_label = "FLIP" if FLIP_FLAGS[idx] else "orig"
    axes[idx].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    axes[idx].set_title(f"Img{idx+1} ({orient_label}) — точек: {n_shown}",
                        fontsize=14)
    axes[idx].axis("off")

plt.suptitle(f"LoFTR: {title_suffix}", fontsize=16)
plt.tight_layout()
plt.show()

## Результаты: граф общих точек

### Попарный LoFTR с фиксированными ориентациями

| Пара | rel | conf>0.4 | RANSAC |
|---|---|---|---|
| (1,2) | flip | 115 | 41 |
| (1,3) | flip | 59 | 26 |
| (1,4) | orig | 96 | 43 |
| (1,5) | flip | 78 | 28 |
| (1,6) | orig | 49 | 21 |
| (2,3) | orig | 119 | 51 |
| (2,4) | flip | 99 | 40 |
| (2,5) | orig | 91 | 34 |
| (2,6) | flip | 68 | 25 |
| (3,4) | flip | 55 | 21 |
| (3,5) | orig | 57 | 19 |
| (3,6) | flip | 76 | 32 |
| (4,5) | flip | 58 | 22 |
| (4,6) | orig | 60 | 24 |
| (5,6) | flip | 81 | 27 |

**Всего RANSAC-inliers: 454** (≈30 матчей на пару в среднем). Доля выживших
после RANSAC: 40–65 % от `conf>0.4` — это выше случайного уровня, значит
большинство матчей геометрически согласованы.

### Кластеризация и граф

- 546 узлов в графе (кластеров по всем изображениям).
- 235 треков после Union-Find.

### Распределение треков по покрытию

| Покрытие | Число треков |
|---|---|
| 2 из 6 | 197 |
| 3 из 6 | 31 |
| 4 из 6 | 4 |
| 5 из 6 | 2 |
| **6 из 6** | **1** |

**Треков с прямыми связями для всех пар: 0.**

### Интерпретация

**Формально один трек покрывает все 6 изображений**, но при верификации
через прямые рёбра он не проходит: где-то в цепочке отсутствует прямая
связь между парой изображений (например, между img1 и img6 нет матча,
хотя обе точки входят в трек). Это классический **«дрейф» трека**:
A–B–C–D–E–F связаны последовательно, но полной клики нет.

**Резкое падение числа треков с ростом покрытия** — ключевой признак:

- 197 треков покрывают только 2 изображения — это попарные матчи,
  которые не подтверждаются третьим изображением.
- 31 трек на 3 изображениях — уже интереснее, но всё ещё мало.
- 4 + 2 + 1 = 7 треков с покрытием ≥4 — единичные случаи.
- **Ни одного устойчивого трека на все 6.**

Если бы в данных была хотя бы одна общая структура (например, «сложный
глаз» у всех особей в одном месте), мы бы увидели **десятки** треков на
6 изображениях, каждый с прямыми рёбрами для всех 15 пар. Вместо этого
видим единичные цепочки, которые рвутся при верификации.

### Визуализация

Показаны 7 треков с покрытием ≥4 из 6 (в канонических ориентациях).
Обратите внимание: даже для этих «лучших» треков номера стоят на
**разных анатомических структурах** на разных изображениях — они не
соответствуют одной и той же гомологичной точке.

### Причина

LoFTR уверенно сопоставляет **похожие по форме** структуры (глаз ↔ глаз,
антеннула ↔ антеннула, край створки ↔ край створки), но эти соответствия
**локальны** и не образуют глобально согласованной картины. При переходе
от пары к цепочке из 6 изображений накапливаются ошибки: если на каждой
паре 20 % матчей ложные, за 5 переходов вероятность сохранить «правильную»
связь падает до 0.8⁵ ≈ 33 %. На реальных данных с разными особями
вероятность ещё ниже.

### Итог

**Автоматический поиск общих ключевых точек через LoFTR не даёт
устойчивого результата на этом наборе.** Нейросеть существенно превосходит
классические методы на уровне попарного матчинга (75–160 матчей против
0–3), но при объединении в треки через 6 изображений связи распадаются.
Причина — та же, что и у AKAZE: сравниваются разные особи одного вида,
у которых гомологичные структуры имеют общую форму, но различаются на
уровне пиксельной текстуры. LoFTR «узнаёт» форму, но не может надёжно
привязать её к одной и той же физической точке на всех 6 снимках.

**Решение задачи требует семантического подхода с явной разметкой** —
ручного определения 8–10 морфологических ориентиров (сложный глаз,
основание антеннулы, края створки, мукро, постабдомен) на каждом
изображении.